# Dataset B — XGBoost final reporting

Notebook cung cấp một publication helper không tự chạy; các reporting cells chỉ đọc package được tạo từ frozen XGB-10 evidence. Không có bước huấn luyện, refit, inference, chọn lại threshold hoặc dùng historical XGBoost result làm kết quả cuối.

## Final lifecycle stage map

- **XGB-07 — training:** Train-only cross-validation và fresh full-Train fit; được thực thi ngoài notebook này.
- **XGB-08 — Validation threshold:** một lần inference trên Validation và khóa threshold; được thực thi ngoài notebook này.
- **XGB-09 — Test evaluation:** một lần inference trên Test bằng frozen threshold và tạo evaluation evidence; được thực thi ngoài notebook này.
- **XGB-10 — final review:** review chỉ đọc đối với evidence XGB-07/08/09 và đóng băng bundle đã PASS.
- **XGB-11 — reporting:** tạo và validate reporting package chỉ từ frozen XGB-10 evidence; không nối historical result vào final reporting.
- **XGB-12 — publication:** publication helper chỉ được định nghĩa tại đây; notebook không tự gọi helper hoặc chạy publication.

Một `run_id` duy nhất phải được giữ nguyên qua mọi stage và ba root preprocessing/working/final. Placeholder chỉ được thay bằng exact final run ID sau khi XGB-01 preflight xác nhận ID chưa tồn tại. Không xóa partial artifact để chạy lại cùng ID, không tự tạo attempt khác, và không vá source/config/test sau khi XGB-00V PASS. Nếu execution phát hiện lỗi source hoặc contract, dừng toàn bộ run và giữ evidence thất bại.

## Publication boundary

Hàm publication bên dưới không tự chạy. Trước khi di chuyển, nó validate toàn bộ working bundle và reporting package, từ chối overwrite, sau đó thực hiện một atomic directory operation và validate lại reporting package tại final destination.

In [ ]:
import os
from pathlib import Path

from src.dataset_b.reporting import validate_reporting_directory
from src.dataset_b.review.xgb_final_review import review_xgb_final

def publish_xgboost_final(
    run_id: str,
    *,
    working_root: str | Path,
    final_root: str | Path,
    bundle_validator=review_xgb_final,
    reporting_validator=validate_reporting_directory,
) -> Path:
    if not isinstance(run_id, str) or not run_id or run_id in {'.', '..'}:
        raise ValueError('Publication requires an explicit valid run_id.')
    working_root = Path(working_root).resolve(strict=True)
    final_root = Path(final_root).resolve(strict=True)
    working_dir = (working_root / run_id).resolve(strict=True)
    destination = final_root / run_id
    if working_dir.parent != working_root:
        raise ValueError('Working bundle escaped the authorized root.')
    if destination.exists():
        raise FileExistsError(f'Final XGBoost destination already exists: {destination}')
    if working_dir.stat().st_dev != final_root.stat().st_dev:
        raise OSError('Atomic publication requires working and final roots on the same filesystem.')

    bundle = bundle_validator(run_id, run_dir=working_dir)
    working_summary = reporting_validator(working_dir / 'reporting')
    if bundle.get('status') != 'PASS' or bundle.get('run_id') != run_id:
        raise ValueError('Working XGBoost bundle is incomplete or misaligned.')
    if (
        working_summary.get('reporting_status') != 'COMPLETE'
        or working_summary.get('run_id') != run_id
        or working_summary.get('model_key') != 'xgboost'
    ):
        raise ValueError('Working XGBoost reporting package is incomplete or misaligned.')

    # Validators return materialized values and close their own file contexts before this operation.
    os.replace(working_dir, destination)
    final_summary = reporting_validator(destination / 'reporting')
    if (
        final_summary.get('reporting_status') != 'COMPLETE'
        or final_summary.get('run_id') != run_id
        or final_summary.get('model_key') != 'xgboost'
    ):
        raise ValueError('Published XGBoost reporting package failed final revalidation.')
    return destination

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config/dataset_b/model_protocol_v1.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate the Dataset B project root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_b.reporting import validate_reporting_directory

RUN_ID = 'SET_XGBOOST_FINAL_RUN_ID'
if RUN_ID.startswith('SET_'):
    raise RuntimeError('Set RUN_ID to the approved XGBoost final run before rendering.')
REPORTING_DIR = PROJECT_ROOT / 'artifacts/dataset_b/xgboost/final' / RUN_ID / 'reporting'
report_summary = validate_reporting_directory(REPORTING_DIR)
if report_summary['model_key'] != 'xgboost' or report_summary.get('protocol_model_key') != 'xgb':
    raise ValueError('Reporting package is not the approved XGBoost key mapping.')
print(f"Loaded frozen XGBoost reporting package: {RUN_ID}")

## Frozen lineage and results

In [ ]:
lineage = pd.DataFrame([
    {'Field': 'Reporting model key', 'Value': report_summary['model_key']},
    {'Field': 'Protocol model key', 'Value': report_summary['protocol_model_key']},
    {'Field': 'run_id', 'Value': report_summary['run_id']},
    {'Field': 'Feature version', 'Value': report_summary['feature_version']},
    {'Field': 'Split version', 'Value': report_summary['split_version']},
    {'Field': 'Preprocessing version', 'Value': report_summary['preprocessing_version']},
])
cv_folds = pd.read_csv(REPORTING_DIR / 'cv_fold_metrics.csv')
rq1_metrics = pd.read_csv(REPORTING_DIR / 'rq1_metrics_long.csv')
confusion = pd.read_csv(REPORTING_DIR / 'confusion_matrix.csv')
calibration = pd.read_csv(REPORTING_DIR / 'calibration_curve.csv')
display(lineage)
display(cv_folds)
display(rq1_metrics)
display(confusion)
display(calibration)

## Reporting boundary

Mọi bảng trong notebook đến từ sáu reporting files đã validate. Kết quả lịch sử không được nối vào summary này; so sánh sáu model vẫn là một gate riêng.